# Stage C — Feedback transducers (partially observable, modified-operator MSP)

Two agents interact bidirectionally (from `explorer.html` feedback mode, Barnett & Crutchfield
2015 Corollary 1):

- **Agent T** (states R): receives input `x`, emits `y` — kernel `T^(y|x)`.
- **Agent U** (states S): receives input `y`, emits `x` — kernel `U^(x|y)`.
- Each step they **exchange**: U emits `x`, T reads `x` and emits `y`, both update. So T's output
  drives U and U's output drives T (a closed loop).

The transformer is trained on the observed **exchange token** `(x, y)` (vocab `n_x*n_y`). By
Corollary 1 the two agents' beliefs evolve **independently**, so we probe three targets:

- **T-belief** = agent T's posterior over `R` (dim `n_R`).
- **U modified-operator belief** = agent U's posterior over the joint `(S, X)` where `X` is U's
  own fed-back output (dim `n_S*n_x`) — *this is the "modified-operator MSP" the mentor referred to
  as the partially-observable case*.
- **joint** = concatenation of the two.

Crucially, both beliefs are **deterministically recomputable from the observed `(x,y)` stream**
(verified), so the probe targets are well-defined for an external observer / the transformer.

**Constraint:** keep U small so the modified-operator dim `n_S*n_x` stays interpretable (≤ 4 ⇒
U = IID or SNS/Fractal2 with 2 states, 2 outputs). torch + numpy only. ~30–60 min / 1M steps on A100.

In [ ]:
!pip install -q transformer-lens 2>&1 | tail -1
import sys, torch
print(f'Python {sys.version.split()[0]}  |  PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import warnings, json, os, zipfile, gc
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm   # plain text bar: renders reliably in the Colab/Cursor cell output
from transformer_lens import HookedTransformer, HookedTransformerConfig

warnings.filterwarnings('ignore')
if torch.cuda.is_available():
    torch.set_float32_matmul_precision('high')

# =====================================================================
#  CONFIG  -- feedback loop  T (states R) <-> U (states S)
# =====================================================================
# T reads x, emits y.  U reads y, emits x.  T's arch/params define T^(y|x); U's define U^(x|y).
T_ARCH = 'sns'        # 'iid' | 'sns' | 'fractal2' | 'mess3'   (agent T, states R)
U_ARCH = 'sns'        # keep U small (n_S*n_x <= 4): 'iid' | 'sns' | 'fractal2'

# one parameter set per INPUT symbol of each agent (T's inputs = x in [0,n_x); U's inputs = y in [0,n_y))
T_SNS_P      = [0.35, 0.80]                       # SNS self-loop per input x
U_SNS_P      = [0.50, 0.60]                       # SNS self-loop per input y
T_FRACTAL2   = [(0.5, 0.25, 0.25), (0.5, 0.75, 0.75)]
U_FRACTAL2   = [(0.5, 0.25, 0.25), (0.5, 0.75, 0.75)]
T_MESS3      = [(0.05, 0.85), (0.05, 0.65)]
IID_P        = [0.4, 0.7]                          # IID emit-1 prob per input symbol

N_X = 2   # T's input alphabet = U's output alphabet
N_Y = 2   # T's output alphabet = U's input alphabet

# ---- Transformer architecture ----
N_CTX = 50; D_MODEL = 64; D_HEAD = 8; N_HEADS = 1; N_LAYERS = 4; D_MLP = 256
# ---- Training ----
BATCH_SIZE = 64; LEARNING_RATE = 0.01; SEQUENCE_LEN = N_CTX + 1
NUM_STEPS = 1_000_000; SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ANALYSIS_SEQS = 50_000

# ---- Free-tier Colab (T4) safety: caps steps + analysis size (OOM / idle disconnect). ----
#      Set False on Colab Pro / A100 / L4 for the full 1M-step run.
COLAB_FREE_T4 = True
if COLAB_FREE_T4:
    NUM_STEPS = 100_000
    ANALYSIS_SEQS = 10_000

CKPT_NAME = f'feedback_ckpt_T{T_ARCH}_U{U_ARCH}'   # unique per run (no overwrite)
SAVE_TO_DRIVE = True
DRIVE_DIR = '/content/drive/MyDrive/transducer_runs'
FIG_DIR = f'{CKPT_NAME}/figures'; os.makedirs(FIG_DIR, exist_ok=True)
print(f'T={T_ARCH} U={U_ARCH}  n_x={N_X} n_y={N_Y}  device={DEVICE}')
print(f'COLAB_FREE_T4={COLAB_FREE_T4}  ->  NUM_STEPS={NUM_STEPS:,}  ANALYSIS_SEQS={ANALYSIS_SEQS:,}')
print(f'checkpoint: {CKPT_NAME}.zip   SAVE_TO_DRIVE={SAVE_TO_DRIVE} -> {DRIVE_DIR}')

## 1. Build the feedback loop + belief recompute

Port of `explorer.html` `sampleFeedback`. `T^(y|x)` and `U^(x|y)` are per-input transducer
kernels. `feedback_sample` runs the coupled generative loop to produce observed exchange tokens
`k = x*n_y + y`; `feedback_beliefs` recomputes agent T's belief over `R` and agent U's
modified-operator belief over `(S,X)` deterministically from those tokens, plus the analytical
next-token distribution `P_t` (used for the entropy floor and the `d_mu` predictive metric).

In [ ]:
# ---- per-input transducer kernel builders (agent kernels), ported from explorer.html ----
def _sns(p):
    return np.array([[[p, 1 - p], [0.0, p]], [[0.0, 0.0], [1 - p, 0.0]]])
def _fractal2(b, p, q):
    return np.array([[[b, 0.0], [(1 - b) * q, (1 - b) * (1 - q)]],
                     [[(1 - b) * (1 - p), (1 - b) * p], [0.0, b]]])
def _iid(p1):
    return np.array([[[1 - p1]], [[p1]]])      # 1 state, 2 outputs

def build_agent(arch, n_in, sns_p, fr, mess, iidp):
    out = []
    for i in range(n_in):
        if arch == 'iid':        out.append(_iid(iidp[i % len(iidp)]))
        elif arch == 'sns':      out.append(_sns(sns_p[i % len(sns_p)]))
        elif arch == 'fractal2': out.append(_fractal2(*fr[i % len(fr)]))
        elif arch == 'mess3':
            x, a = mess[i % len(mess)]; b, y = (1 - a) / 2, 1 - 2 * x
            ay, bx, by, ax = a*y, b*x, b*y, a*x
            out.append(np.array([[[ay,bx,bx],[ax,by,bx],[ax,bx,by]],
                                 [[by,ax,bx],[bx,ay,bx],[bx,ax,by]],
                                 [[by,bx,ax],[bx,by,ax],[bx,bx,ay]]]))
        else: raise ValueError(arch)
    return np.stack(out)

# T reads x (n_x inputs), emits y (n_y outputs); Tperx[x] has shape (n_y, n_R, n_R)
Tperx = build_agent(T_ARCH, N_X, T_SNS_P, T_FRACTAL2, T_MESS3, IID_P)   # (n_x, n_y, n_R, n_R)
# U reads y (n_y inputs), emits x (n_x outputs); Uupery[y] has shape (n_x, n_S, n_S)
Uupery = build_agent(U_ARCH, N_Y, U_SNS_P, U_FRACTAL2, T_MESS3, IID_P)  # (n_y, n_x, n_S, n_S)
n_x, n_y = N_X, N_Y
n_R, n_S = Tperx.shape[2], Uupery.shape[2]
n_SX = n_S * n_x
VOCAB_SIZE = n_x * n_y
T_rowsumR = Tperx.sum(axis=3)      # (n_x, n_y, n_R)  = Σ_{r'} T^(y|x)[r,r']

def _stationary(M):
    v = np.ones(M.shape[0]) / M.shape[0]
    for _ in range(3000):
        w = v @ M; w = w / w.sum()
        if np.abs(w - v).sum() < 1e-14: break
        v = w
    return v
# Initial T-belief: stationary of the input-averaged T dynamics (Σ_x Σ_y T^(y|x) / n_x).
RHO_R0 = _stationary(sum(Tperx[x][y] for x in range(n_x) for y in range(n_y)) / n_x)

def _predicted_token_dist(rhoR, rhoSX):
    """P(x,y | beliefs) as a (b, VOCAB) matrix; k = x*n_y + y."""
    b = rhoR.shape[0]
    xMarg = rhoSX.reshape(b, n_S, n_x).sum(1)
    xMarg = xMarg / np.clip(xMarg.sum(1, keepdims=True), 1e-30, None)
    ydist = np.einsum('br,xyr->bxy', rhoR, T_rowsumR)            # (b, n_x, n_y)
    ydist = ydist / np.clip(ydist.sum(2, keepdims=True), 1e-30, None)
    P = xMarg[:, :, None] * ydist                               # (b, n_x, n_y)
    return P.reshape(b, VOCAB_SIZE)

def _update(rhoR, rhoSX, xObs, yObs):
    """Deterministic belief update given observed (xObs, yObs) per batch element."""
    b = rhoR.shape[0]
    Tsel = Tperx[xObs, yObs]                                    # (b, n_R, n_R)
    rhoR = np.einsum('br,brR->bR', rhoR, Tsel)
    rhoR = rhoR / np.clip(rhoR.sum(1, keepdims=True), 1e-30, None)
    old_s = rhoSX.reshape(b, n_S, n_x)[np.arange(b), :, xObs]   # (b, n_S) mass at (s, xObs)
    U_y = Uupery[yObs]                                          # (b, n_x, n_S, n_S) = [xp,s,sp]
    newSX = np.einsum('bs,bxsp->bpx', old_s, U_y)               # (b, n_S(sp), n_x(xp))
    newSX = newSX.reshape(b, n_SX)
    newSX = newSX / np.clip(newSX.sum(1, keepdims=True), 1e-30, None)
    return rhoR, newSX

def feedback_sample(rng, n_seqs, seq_len):
    """Coupled generative loop -> observed composite tokens k=x*n_y+y. (n_seqs, seq_len)."""
    rhoR = np.tile(RHO_R0, (n_seqs, 1)); rhoSX = np.full((n_seqs, n_SX), 1.0 / n_SX)
    toks = np.zeros((n_seqs, seq_len), dtype=np.int64)
    for t in range(seq_len):
        P = _predicted_token_dist(rhoR, rhoSX)                  # (b, VOCAB)
        cum = P.cumsum(1); u = rng.random(n_seqs)[:, None]
        k = (u >= cum).sum(1); k = np.clip(k, 0, VOCAB_SIZE - 1)
        toks[:, t] = k
        xObs, yObs = k // n_y, k % n_y
        rhoR, rhoSX = _update(rhoR, rhoSX, xObs, yObs)
    return toks

def feedback_beliefs(toks):
    """Recompute (T_belief over R, U modified-op belief over (S,X), predicted token dist P)."""
    n_seqs, seq_len = toks.shape
    rhoR = np.tile(RHO_R0, (n_seqs, 1)); rhoSX = np.full((n_seqs, n_SX), 1.0 / n_SX)
    Rb = np.zeros((n_seqs, seq_len, n_R), np.float32)
    SXb = np.zeros((n_seqs, seq_len, n_SX), np.float32)
    Pt = np.zeros((n_seqs, seq_len, VOCAB_SIZE), np.float32)
    for t in range(seq_len):
        k = toks[:, t]; xObs, yObs = k // n_y, k % n_y
        rhoR, rhoSX = _update(rhoR, rhoSX, xObs, yObs)
        Rb[:, t] = rhoR; SXb[:, t] = rhoSX
        Pt[:, t] = _predicted_token_dist(rhoR, rhoSX)
    return Rb, SXb, Pt

# ---- entropy rate of the observed exchange process (mean per-step token entropy) ----
def feedback_entropy_rate(n_steps=100_000, burn=2000, seed=0):
    rng = np.random.default_rng(seed)
    rhoR = RHO_R0[None].copy(); rhoSX = np.full((1, n_SX), 1.0 / n_SX)
    h = 0.0; cnt = 0
    for t in range(n_steps):
        P = _predicted_token_dist(rhoR, rhoSX)[0]
        if t >= burn:
            h += -np.sum(P * np.log(P + 1e-30)); cnt += 1
        cum = P.cumsum(); k = int((rng.random() >= cum).sum()); k = min(k, VOCAB_SIZE - 1)
        xObs, yObs = np.array([k // n_y]), np.array([k % n_y])
        rhoR, rhoSX = _update(rhoR, rhoSX, xObs, yObs)
    return h / cnt

ENTROPY_RATE = feedback_entropy_rate()
print(f'n_R={n_R} n_S={n_S} n_x={n_x} n_y={n_y}  |  vocab(exchange)={VOCAB_SIZE}  '
      f'|  U modified-op dim (S,X)={n_SX}')
print(f'entropy rate of exchange process: {ENTROPY_RATE:.4f} nats  |  log(vocab)={np.log(VOCAB_SIZE):.4f}')

## 2. Belief manifolds — T-agent (over R) and U modified-operator MSP (over S,X)

In [ ]:
def embed_simplex(P):
    n = P.shape[1]
    if n == 1: return np.zeros((len(P), 2))
    if n == 2: return np.column_stack([P[:, 0], np.zeros(len(P))])
    if n == 3:
        return P @ np.array([[0.0, 0.0], [1.0, 0.0], [0.5, np.sqrt(3) / 2]])
    Pc = P - P.mean(0); _, _, Vt = np.linalg.svd(Pc, full_matrices=False)
    return Pc @ Vt[:2].T

_toks = feedback_sample(np.random.default_rng(SEED + 5), 400, 200)
_Rb, _SXb, _ = feedback_beliefs(_toks)
Rb_all = _Rb.reshape(-1, n_R); SXb_all = _SXb.reshape(-1, n_SX)
print(f'distinct T-belief (R) states:            {np.unique(Rb_all.round(4), axis=0).shape[0]}')
print(f'distinct U modified-op (S,X) states:     {np.unique(SXb_all.round(4), axis=0).shape[0]}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5.4))
eR = embed_simplex(Rb_all)
if n_R == 2:
    axes[0].scatter(Rb_all[:, 0], np.random.default_rng(0).uniform(-.4, .4, len(Rb_all)), s=4, alpha=0.3)
    axes[0].set_xlim(-0.02, 1.02); axes[0].set_yticks([]); axes[0].set_xlabel('T-belief (R state 0)')
else:
    axes[0].scatter(eR[:, 0], eR[:, 1], s=3, alpha=0.3); axes[0].set_aspect('equal')
axes[0].set_title(f'Agent T belief manifold (over R, dim {n_R})')

eSX = embed_simplex(SXb_all)
axes[1].scatter(eSX[:, 0], eSX[:, 1], s=3, alpha=0.3, color='#8E24AA')
axes[1].set_title(f'Agent U modified-operator MSP (over S,X, dim {n_SX})')
axes[1].set_aspect('equal')
plt.suptitle('Feedback belief manifolds (Corollary 1: evolve independently)', y=1.02)
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/fig_feedback_manifolds.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# ---- Transformer (vocab = exchange token x*n_y+y) + data / probe helpers ----
torch.manual_seed(SEED)
def make_model(seed, n_heads):
    cfg = HookedTransformerConfig(
        n_ctx=N_CTX, d_model=D_MODEL, d_head=D_HEAD, n_heads=n_heads, n_layers=N_LAYERS,
        d_mlp=D_MLP, d_vocab=VOCAB_SIZE, act_fn='relu', normalization_type='LN', device=DEVICE, seed=seed)
    return HookedTransformer(cfg), cfg
try:
    model, tl_cfg = make_model(SEED, N_HEADS); ACTUAL_N_HEADS = N_HEADS
except Exception:
    ACTUAL_N_HEADS = 8; model, tl_cfg = make_model(SEED, ACTUAL_N_HEADS)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}  |  vocab={VOCAB_SIZE}  |  n_heads={ACTUAL_N_HEADS}')

all_layer_keys = [f'blocks.{l}.hook_resid_post' for l in range(N_LAYERS)]

def generate_batch(rng_obj):
    seqs = feedback_sample(rng_obj, BATCH_SIZE, SEQUENCE_LEN)
    return (torch.tensor(seqs[:, :-1], device=DEVICE),
            torch.tensor(seqs[:, 1:], device=DEVICE, dtype=torch.long))

def belief_targets(toks):
    """Returns dict of probe targets flattened over (seq, pos)."""
    Rb, SXb, _ = feedback_beliefs(toks)
    Rf = Rb.reshape(-1, n_R); SXf = SXb.reshape(-1, n_SX)
    return {'T_belief': Rf, 'U_modified_op': SXf, 'joint': np.concatenate([Rf, SXf], axis=1)}

def extract_activations(net, seqs, batch=2048):
    net.eval(); seqs_t = torch.tensor(seqs, dtype=torch.long); acc = {k: [] for k in all_layer_keys}
    with torch.no_grad():
        for s in range(0, len(seqs), batch):
            _, cache = net.run_with_cache(seqs_t[s:s + batch].to(DEVICE))
            for k in all_layer_keys: acc[k].append(cache[k].detach().cpu().numpy())
            del cache
    return {k: np.concatenate(v, 0).reshape(-1, D_MODEL) for k, v in acc.items()}

def ols_fit_eval(X_data, Y_data):
    X = np.column_stack([np.ones(X_data.shape[0]), X_data]).astype(np.float64)
    Y = np.atleast_2d(Y_data.astype(np.float64));  Y = Y.T if Y.shape[0] != X.shape[0] else Y
    beta, _, _, _ = np.linalg.lstsq(X, Y, rcond=None); resid = X @ beta - Y
    ss_res = np.sum(resid ** 2); ss_tot = np.sum((Y - Y.mean(0)) ** 2)
    return {'mse': float(np.mean(np.sum(resid ** 2, 1))),
            'r2': float(1 - ss_res / ss_tot) if ss_tot > 1e-12 else float('nan')}

def run_checkpoint_analysis(n_seqs=5000):
    seqs = feedback_sample(np.random.default_rng(SEED + 777), n_seqs, N_CTX)
    tgt = belief_targets(seqs)['U_modified_op']
    acts = extract_activations(model, seqs)[all_layer_keys[-1]]
    return ols_fit_eval(acts, tgt)
print('helpers defined.')

## 3. Train OR load  (Option A = train; Option B = load `feedback_checkpoint.zip`)

In [ ]:
# ---- OPTION B: load checkpoint (skip training) ----
save_dir = 'feedback_checkpoint'; zip_path = 'feedback_checkpoint.zip'
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zf: zf.extractall('.')
    print(f'Extracted {save_dir}/')
if os.path.exists(f'{save_dir}/model_weights.pt'):
    model.load_state_dict(torch.load(f'{save_dir}/model_weights.pt', map_location=DEVICE)); model.eval()
    th = np.load(f'{save_dir}/training_history.npz'); losses = list(th['losses'])
    checkpoint_results = {int(s): {'mse': float(m), 'r2': float(r)}
                          for s, m, r in zip(th['checkpoint_steps'], th['checkpoint_mses'], th['checkpoint_r2s'])}
    print(f'Loaded. {len(losses):,} steps, final loss={losses[-1]:.4f}')
else:
    print('No checkpoint -- run the training cell (Option A).')

In [ ]:
# ---- OPTION A: train from scratch ----
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE)
rng = np.random.default_rng(SEED)
checkpoints = [c for c in [0, 10_000, 50_000, 100_000, 250_000, 500_000, 750_000, NUM_STEPS] if c <= NUM_STEPS]
checkpoint_results = {}; losses = []; model.train()
pbar = tqdm(range(NUM_STEPS + 1), desc='Training')
for step in pbar:
    if step in checkpoints:
        sc = run_checkpoint_analysis(); checkpoint_results[step] = sc
        pbar.set_postfix(mse=f"{sc['mse']:.5f}", r2=f"{sc['r2']:.4f}"); model.train()
    if step == 0: continue
    inputs, labels = generate_batch(rng)
    loss = loss_fn(model(inputs).reshape(-1, VOCAB_SIZE), labels.reshape(-1))
    optimizer.zero_grad(); loss.backward(); optimizer.step(); losses.append(loss.item())
    if step % 5000 == 0: pbar.set_postfix(loss=f'{loss.item():.4f}')
print(f'\nFinal loss (smoothed 10k): {np.mean(losses[-10000:]):.4f}  |  entropy floor: {ENTROPY_RATE:.4f}')

In [ ]:
# ---- training curve + predictive metric (d_mu KL + entropy gap) ----
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
ax = axes[0]; ax.plot(losses, lw=0.2, alpha=0.3, label='per-step')
w = min(2000, len(losses) // 5) if len(losses) > 10 else 1
if w > 1:
    sm = np.convolve(losses, np.ones(w) / w, mode='valid'); ax.plot(np.arange(w - 1, w - 1 + len(sm)), sm, lw=1.2, label=f'{w}-avg')
ax.axhline(ENTROPY_RATE, color='red', ls='--', lw=0.9, label=f'entropy rate {ENTROPY_RATE:.3f}')
ax.set_xlabel('step'); ax.set_ylabel('loss (nats)'); ax.set_title('Training loss'); ax.legend()
ax = axes[1]; steps = sorted(checkpoint_results)
ax.plot(steps, [checkpoint_results[s]['mse'] for s in steps], 'o-'); ax.set_yscale('log')
ax.set_xlabel('step'); ax.set_ylabel('U modified-op MSE'); ax.set_title('U-belief MSE vs training')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/fig_training.png', dpi=150, bbox_inches='tight'); plt.show()

def predictive_dmu(net, n_seqs=2000, seq_len=N_CTX):
    seqs = feedback_sample(np.random.default_rng(SEED + 555), n_seqs, seq_len)
    _, _, P = feedback_beliefs(seqs)
    net.eval(); seqs_t = torch.tensor(seqs, dtype=torch.long); kls, ces = [], []
    with torch.no_grad():
        for s in range(0, n_seqs, 2048):
            Q = torch.softmax(net(seqs_t[s:s + 2048].to(DEVICE)), -1).cpu().numpy()
            Pb = P[s:s + 2048]
            kls.append(np.sum(Pb * (np.log(Pb + 1e-30) - np.log(Q + 1e-30)), -1).reshape(-1))
            nxt = seqs[s:s + 2048, 1:]; qn = Q[:, :-1, :]
            ces.append((-np.log(np.take_along_axis(qn, nxt[:, :, None], -1)[..., 0] + 1e-30)).reshape(-1))
    return float(np.mean(np.concatenate(kls))), float(np.mean(np.concatenate(ces)))

D_MU, MODEL_CE = predictive_dmu(model)
print(f'd_mu (mean KL P||Q) = {D_MU:.5f} nats   model CE = {MODEL_CE:.4f}   '
      f'h_mu = {ENTROPY_RATE:.4f}   gap = {MODEL_CE - ENTROPY_RATE:+.4f}')

## 4. Probes — T-belief, U modified-operator belief, joint (per layer + untrained baseline)

In [ ]:
# ---- analysis data + activations (trained + untrained) ----
print(f'Generating {ANALYSIS_SEQS:,} sequences...')
analysis_seqs = feedback_sample(np.random.default_rng(SEED + 999), ANALYSIS_SEQS, N_CTX)
TARGETS = belief_targets(analysis_seqs)
n_data = TARGETS['T_belief'].shape[0]
print(f'data points: {n_data:,}  |  T_belief({n_R})  U_modified_op({n_SX})  joint({n_R + n_SX})')

print('Extracting trained activations...')
trained_layers = extract_activations(model, analysis_seqs)
concat_acts = np.concatenate([trained_layers[k] for k in all_layer_keys], axis=1)
print('Extracting untrained baseline...')
untrained_model, _ = make_model(SEED + 12345, ACTUAL_N_HEADS)
untrained_layers = extract_activations(untrained_model, analysis_seqs)
untrained_concat = np.concatenate([untrained_layers[k] for k in all_layer_keys], axis=1)
del untrained_model
if DEVICE == 'cuda': torch.cuda.empty_cache()

In [ ]:
layer_labels = [f'L{l}' for l in range(N_LAYERS)] + ['Concat']
def probes_for(acts_by_layer, acts_concat):
    out = {}
    for tname, target in TARGETS.items():
        out[tname] = {}
        for k, lbl in zip(all_layer_keys, layer_labels[:-1]):
            out[tname][lbl] = ols_fit_eval(acts_by_layer[k], target)
        out[tname]['Concat'] = ols_fit_eval(acts_concat, target)
    return out
results = probes_for(trained_layers, concat_acts)
results_untrained = probes_for(untrained_layers, untrained_concat)

print('=' * 78)
print(f'{"target":<16}{"layer":<8}{"trained R2":>12}{"trained MSE":>14}{"untrn R2":>12}')
print('-' * 78)
for tname in TARGETS:
    for lbl in layer_labels:
        t, u = results[tname][lbl], results_untrained[tname][lbl]
        print(f'{tname:<16}{lbl:<8}{t["r2"]:>12.4f}{t["mse"]:>14.6f}{u["r2"]:>12.4f}')
    print()

fig, axes = plt.subplots(1, len(TARGETS), figsize=(5 * len(TARGETS), 4))
for ax, tname in zip(axes, TARGETS):
    tr = [results[tname][l]['mse'] for l in layer_labels]; un = [results_untrained[tname][l]['mse'] for l in layer_labels]
    x = np.arange(len(layer_labels)); ax.bar(x - .2, tr, .4, label='trained', color='#2196F3')
    ax.bar(x + .2, un, .4, label='untrained', color='#FF9800', alpha=.8)
    ax.set_xticks(x); ax.set_xticklabels(layer_labels); ax.set_yscale('log'); ax.set_title(tname); ax.legend(fontsize=8)
plt.suptitle('Feedback probe MSE by layer (trained vs untrained)', y=1.03)
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/fig_probes.png', dpi=150, bbox_inches='tight'); plt.show()

del untrained_layers, untrained_concat
gc.collect()

## 5. Controls (on the U modified-operator belief)

In [ ]:
CTRL = TARGETS['U_modified_op']; N_TRIALS = 50; ctrl_rng = np.random.default_rng(SEED)
full_mse = results['U_modified_op']['Concat']['mse']
untrained_mse = results_untrained['U_modified_op']['Concat']['mse']

def fit_test_mse(tr_i, te_i):
    Xtr = np.column_stack([np.ones(len(tr_i)), concat_acts[tr_i]]).astype(np.float64)
    beta, _, _, _ = np.linalg.lstsq(Xtr, CTRL[tr_i].astype(np.float64), rcond=None)
    Xte = np.column_stack([np.ones(len(te_i)), concat_acts[te_i]]).astype(np.float64)
    return float(np.mean(np.sum((Xte @ beta - CTRL[te_i]) ** 2, 1)))

seq_cv = []
for _ in tqdm(range(N_TRIALS), desc='Seq-CV'):
    perm = ctrl_rng.permutation(ANALYSIS_SEQS); ntr = int(0.2 * ANALYSIS_SEQS)
    tr_i = np.concatenate([np.arange(s * N_CTX, (s + 1) * N_CTX) for s in perm[:ntr]])
    te_i = np.concatenate([np.arange(s * N_CTX, (s + 1) * N_CTX) for s in perm[ntr:]])
    seq_cv.append(fit_test_mse(tr_i, te_i))

shuffle_mse = []; X_all = np.column_stack([np.ones(n_data), concat_acts]).astype(np.float64)
for _ in tqdm(range(N_TRIALS), desc='Shuffle'):
    p = ctrl_rng.permutation(n_data)
    beta, _, _, _ = np.linalg.lstsq(X_all, CTRL[p].astype(np.float64), rcond=None)
    shuffle_mse.append(float(np.mean(np.sum((X_all @ beta - CTRL[p]) ** 2, 1))))

mid = N_CTX // 2
first = np.concatenate([np.arange(s * N_CTX, s * N_CTX + mid) for s in range(ANALYSIS_SEQS)])
second = np.concatenate([np.arange(s * N_CTX + mid, (s + 1) * N_CTX) for s in range(ANALYSIS_SEQS)])
tsplit = 0.5 * (fit_test_mse(first, second) + fit_test_mse(second, first))

controls = {'full_mse': full_mse, 'untrained_mse': untrained_mse,
            'seq_cv_test_mean': float(np.mean(seq_cv)), 'tsplit_mse': float(tsplit),
            'shuffle_mse_mean': float(np.mean(shuffle_mse))}
print('\n=== Controls (U modified-op belief, concat) ===')
for k, v in controls.items(): print(f'  {k:<20} {v:.6f}')
print(f'  shuffle/full={np.mean(shuffle_mse)/max(full_mse,1e-12):.1f}x  '
      f'untrained/full={untrained_mse/max(full_mse,1e-12):.1f}x  '
      f'seq-CV/full={np.mean(seq_cv)/max(full_mse,1e-12):.2f}x')
fig, ax = plt.subplots(figsize=(8, 4))
lab = ['Full', 'Untrained', 'Seq-CV', 'Tsplit', 'Shuffle']
val = [full_mse, untrained_mse, np.mean(seq_cv), tsplit, np.mean(shuffle_mse)]
ax.bar(lab, val, color=['#2196F3', '#FF9800', '#9C27B0', '#00897B', '#F44336'], alpha=.85)
ax.set_ylabel('MSE'); ax.set_title('Controls: U modified-op belief')
for i, v in enumerate(val): ax.text(i, v, f'{v:.4f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/fig_controls.png', dpi=150, bbox_inches='tight'); plt.show()

del X_all
gc.collect()

## 6. Modified-operator MSP: ground truth vs residual-stream readout

In [ ]:
X_c = np.column_stack([np.ones(n_data), concat_acts]).astype(np.float64)
beta_u, _, _, _ = np.linalg.lstsq(X_c, TARGETS['U_modified_op'].astype(np.float64), rcond=None)
pred_u = np.clip((X_c @ beta_u).astype(np.float32), 0, None)
true_u = TARGETS['U_modified_op']

idx = np.random.default_rng(0).choice(n_data, min(40_000, n_data), replace=False)
# shared PCA basis from ground-truth modified-op beliefs for a fair visual comparison
base = true_u[idx] - true_u[idx].mean(0)
_, _, Vt = np.linalg.svd(base, full_matrices=False); pcs = Vt[:2].T
et = base @ pcs; ep = (pred_u[idx] - true_u[idx].mean(0)) @ pcs

fig, axes = plt.subplots(1, 2, figsize=(13, 5.6))
axes[0].scatter(et[:, 0], et[:, 1], s=2, alpha=0.2, color='#8E24AA', rasterized=True)
axes[0].set_title(f'Ground-truth U modified-op MSP (dim {n_SX})'); axes[0].set_aspect('equal')
axes[1].scatter(ep[:, 0], ep[:, 1], s=2, alpha=0.2, color='#D84315', rasterized=True)
axes[1].set_title(f"Learned readout (R2={results['U_modified_op']['Concat']['r2']:.3f})"); axes[1].set_aspect('equal')
plt.suptitle('Modified-operator MSP: ground truth vs residual stream (shared PCA basis)', y=1.02)
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/fig_msp_vs_learned.png', dpi=150, bbox_inches='tight'); plt.show()

## 7. Save checkpoint

In [ ]:
save_dir = CKPT_NAME; os.makedirs(save_dir, exist_ok=True)
if not os.path.exists(f'{save_dir}/model_weights.pt'):
    torch.save(model.state_dict(), f'{save_dir}/model_weights.pt')

def _j(v): return None if (isinstance(v, float) and np.isnan(v)) else v
config = {
    'mode': 'feedback', 't_arch': T_ARCH, 'u_arch': U_ARCH,
    'n_x': n_x, 'n_y': n_y, 'n_R': n_R, 'n_S': n_S, 'n_SX': n_SX, 'vocab': VOCAB_SIZE,
    'n_ctx': N_CTX, 'd_model': D_MODEL, 'n_layers': N_LAYERS, 'n_heads': ACTUAL_N_HEADS,
    'seed': SEED, 'num_steps_trained': NUM_STEPS, 'entropy_rate': float(ENTROPY_RATE),
    'final_loss_smoothed': float(np.mean(losses[-10000:])), 'd_mu': float(D_MU), 'model_ce': float(MODEL_CE),
    'T_belief_r2_concat': _j(results['T_belief']['Concat']['r2']),
    'U_modified_op_r2_concat': _j(results['U_modified_op']['Concat']['r2']),
    'joint_r2_concat': _j(results['joint']['Concat']['r2']),
}
with open(f'{save_dir}/config.json', 'w') as f: json.dump(config, f, indent=2)
np.savez_compressed(f'{save_dir}/training_history.npz', losses=np.array(losses),
    checkpoint_steps=np.array(sorted(checkpoint_results)),
    checkpoint_mses=np.array([checkpoint_results[s]['mse'] for s in sorted(checkpoint_results)]),
    checkpoint_r2s=np.array([checkpoint_results[s]['r2'] for s in sorted(checkpoint_results)]))
probe_data = {}
for tn in results:
    for lbl in results[tn]:
        probe_data[f'{tn}_{lbl}_r2'] = results[tn][lbl]['r2']; probe_data[f'{tn}_{lbl}_mse'] = results[tn][lbl]['mse']
        probe_data[f'untrained_{tn}_{lbl}_r2'] = results_untrained[tn][lbl]['r2']
np.savez_compressed(f'{save_dir}/probe_results.npz', **probe_data)
with open(f'{save_dir}/controls.json', 'w') as f: json.dump(controls, f, indent=2)
zip_name = f'{CKPT_NAME}.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, fns in os.walk(save_dir):
        for fn in fns: zf.write(os.path.join(root, fn))
print(f'Saved {zip_name}')
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive; drive.mount('/content/drive')
        os.makedirs(DRIVE_DIR, exist_ok=True)
        import shutil; shutil.copy(zip_name, f'{DRIVE_DIR}/{zip_name}')
        print(f'Copied to Google Drive: {DRIVE_DIR}/{zip_name}')
    except Exception as e:
        print(f'Drive save skipped ({e}); zip is at /content/{zip_name}')
else:
    try:
        from google.colab import files; files.download(zip_name)
    except ImportError:
        print('Not in Colab -- zip saved locally.')